# Data Loading

In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
%matplotlib inline

In [2]:
bmx_data   = pd.read_sas("../data/P_BMX.xpt", format="xport")
demo_data  = pd.read_sas("../data/P_DEMO.xpt", format="xport")
diq_data   = pd.read_sas("../data/P_DIQ.xpt", format="xport")
ghb_data   = pd.read_sas("../data/P_GHB.xpt", format="xport")
glu_data   = pd.read_sas("../data/P_GLU.xpt", format="xport")
bp_data    = pd.read_sas("../data/P_BPXO.XPT", format="xport")
hdl_data   = pd.read_sas("../data/P_HDL.XPT", format="xport")
bio_data   = pd.read_sas("../data/P_BIOPRO.XPT", format="xport")
albcr_data = pd.read_sas("../data/P_ALB_CR.XPT", format="xport")

# Data Inspection

In [3]:
print("\nBMX columns:\n", bmx_data.columns.tolist())
print("\nDEMO columns:\n", demo_data.columns.tolist())
print("\nDIQ columns:\n", diq_data.columns.tolist())
print("\nGHB columns:\n", ghb_data.columns.tolist())
print("\nGLU columns:\n", glu_data.columns.tolist())
print("\nBP columns:\n", bp_data.columns.tolist())
print("\nHDL columns:\n", hdl_data.columns.tolist())
print("\nBIOPRO columns:\n", bio_data.columns.tolist())
print("\nALB_CR columns:\n", albcr_data.columns.tolist())



BMX columns:
 ['SEQN', 'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD', 'BMIHEAD', 'BMXHT', 'BMIHT', 'BMXBMI', 'BMDBMIC', 'BMXLEG', 'BMILEG', 'BMXARML', 'BMIARML', 'BMXARMC', 'BMIARMC', 'BMXWAIST', 'BMIWAIST', 'BMXHIP', 'BMIHIP']

DEMO columns:
 ['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'DMDBORN4', 'DMDYRUSZ', 'DMDEDUC2', 'DMDMARTZ', 'RIDEXPRG', 'SIALANG', 'SIAPROXY', 'SIAINTRP', 'FIALANG', 'FIAPROXY', 'FIAINTRP', 'MIALANG', 'MIAPROXY', 'MIAINTRP', 'AIALANGA', 'WTINTPRP', 'WTMECPRP', 'SDMVPSU', 'SDMVSTRA', 'INDFMPIR']

DIQ columns:
 ['SEQN', 'DIQ010', 'DID040', 'DIQ160', 'DIQ180', 'DIQ050', 'DID060', 'DIQ060U', 'DIQ070', 'DIQ230', 'DIQ240', 'DID250', 'DID260', 'DIQ260U', 'DIQ275', 'DIQ280', 'DIQ291', 'DIQ300S', 'DIQ300D', 'DID310S', 'DID310D', 'DID320', 'DID330', 'DID341', 'DID350', 'DIQ350U', 'DIQ360', 'DIQ080']

GHB columns:
 ['SEQN', 'LBXGH']

GLU columns:
 ['SEQN', 'WTSAFPRP', 'LBXGLU', 'LBDGLUSI']

BP 

In [4]:
print("Checking for missing values in the important columns:")

print("BMI Missing Values:", bmx_data["BMXBMI"].isnull().sum())
print("HbA1c Missing Values:", ghb_data["LBXGH"].isnull().sum())
print("Diabetes Presence Missing Values:", diq_data["DIQ010"].isnull().sum())
print("SBP1 Missing Values:", bp_data["BPXOSY1"].isnull().sum())
print("SBP2 Missing Values:", bp_data["BPXOSY2"].isnull().sum())
print("SBP3 Missing Values:", bp_data["BPXOSY3"].isnull().sum())
print("HDL Missing Values:", hdl_data["LBDHDD"].isnull().sum())
print("Serum Creatinine Missing Values:", bio_data["LBXSCR"].isnull().sum())
print("Urine Albumin Missing Values:", albcr_data["URXUMA"].isnull().sum())
print("Urine Creatinine Missing Values:", albcr_data["URXUCR"].isnull().sum())


Checking for missing values in the important columns:
BMI Missing Values: 1163
HbA1c Missing Values: 672
Diabetes Presence Missing Values: 0
SBP1 Missing Values: 1304
SBP2 Missing Values: 1329
SBP3 Missing Values: 1370
HDL Missing Values: 1370
Serum Creatinine Missing Values: 934
Urine Albumin Missing Values: 517
Urine Creatinine Missing Values: 518


# Data Preprocessing 

## Data Merging 

In [5]:
bp_data["SBP_mean"] = bp_data[["BPXOSY1", "BPXOSY2", "BPXOSY3"]].mean(axis=1)
albcr_data["ACR"] = albcr_data["URXUMA"] / albcr_data["URXUCR"].replace({0: np.nan})

### Remove unecessary colums 

In [6]:
demo = demo_data[["SEQN","RIDAGEYR","RIAGENDR","RIDRETH3"]].copy()
bmx = bmx_data[["SEQN","BMXBMI"]].copy()
ghb = ghb_data[["SEQN","LBXGH"]].copy()
diq = diq_data[["SEQN","DIQ010"]].copy()
glu = glu_data[["SEQN","LBXGLU"]].copy()
bp = bp_data[["SEQN","SBP_mean"]].copy()
hdl = hdl_data[["SEQN","LBDHDD"]].copy()
bio = bio_data[["SEQN","LBXSCR"]].copy()
albcr = albcr_data[["SEQN","ACR"]].copy()

In [7]:
def dup_count(df, name):
    dups = df["SEQN"].duplicated().sum()
    print(f"{name}: rows={len(df):,} | duplicated SEQN={dups:,}")

dup_count(demo,  "DEMO")
dup_count(bmx,   "BMX")
dup_count(ghb,   "GHB")
dup_count(diq,   "DIQ")
dup_count(glu,   "GLU")
dup_count(bp,    "BPXO")
dup_count(hdl,   "HDL")
dup_count(bio,   "BIOPRO")
dup_count(albcr, "ALB_CR")


DEMO: rows=15,560 | duplicated SEQN=0
BMX: rows=14,300 | duplicated SEQN=0
GHB: rows=10,409 | duplicated SEQN=0
DIQ: rows=14,986 | duplicated SEQN=0
GLU: rows=5,090 | duplicated SEQN=0
BPXO: rows=11,656 | duplicated SEQN=0
HDL: rows=12,198 | duplicated SEQN=0
BIOPRO: rows=10,409 | duplicated SEQN=0
ALB_CR: rows=13,027 | duplicated SEQN=0


### Merging Datasets

In [8]:
data = demo.merge(bmx, on = "SEQN", how = "left")
data = data.merge(ghb, on = "SEQN", how = "left")
data = data.merge(diq, on = "SEQN", how = "left")
data = data.merge(glu, on = "SEQN", how = "left")
data = data.merge(bp, on = "SEQN", how = "left")
data = data.merge(hdl, on = "SEQN", how = "left")
data = data.merge(bio, on = "SEQN", how = "left")
data = data.merge(albcr, on = "SEQN", how = "left")

In [9]:
print("Final Dataset Shape: ", data.shape)
data.head()

Final Dataset Shape:  (15560, 12)


,SEQN,RIDAGEYR,RIAGENDR,RIDRETH3,BMXBMI,LBXGH,DIQ010,LBXGLU,SBP_mean,LBDHDD,LBXSCR,ACR
0,109263.0,2.0,1.0,6.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN
1,109264.0,13.0,2.0,1.0,17.6,5.3,2.0,97.0,108.0,72.0,0.57,NaN
2,109265.0,2.0,1.0,3.0,15.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN
3,109266.0,29.0,2.0,6.0,37.8,5.2,2.0,NaN,99.0,56.0,0.63,0.152778
4,109267.0,21.0,2.0,2.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN


## Making The Label Feature

In [10]:
data = data[data["RIDAGEYR"] >= 18].copy()
has_dx = (data["DIQ010"]==1)
has_high_hba1c = (data["LBXGH"] >= 6.5)

data["diabetes_label"] = (has_dx | has_high_hba1c).astype(int)
data["diabetes_label"].value_counts()

diabetes_label
0    7985
1    1708
Name: count, dtype: int64

## Handling Missing Values 

In [11]:
data.isnull().sum()

SEQN                 0
RIDAGEYR             0
RIAGENDR             0
RIDRETH3             0
BMXBMI             903
LBXGH             1229
DIQ010               0
LBXGLU            5520
SBP_mean          1669
LBDHDD            1395
LBXSCR            1436
ACR                964
diabetes_label       0
dtype: int64

In [ ]:
data["BMXBMI"] = data["BMXBMI"].fillna(data["BMXBMI"].median())
data["LBXGH_missing"] = data["LBXGH"].isna().astype(int)
data["LBXGH"] = data["LBXGH"].fillna(data["LBXGH"].median())
data["SBP_mean"] = data["SBP_mean"].fillna(data["SBP_mean"].median())
data["HDL_missing"] = data["LBDHDD"].isna().astype(int)
data["LBDHDD"] = data["LBDHDD"].fillna(data["LBDHDD"].median())
data["LBXSCR"] = data["LBXSCR"].fillna(data["LBXSCR"].median()) 
data["ACR_missing"] = data["ACR"].isna().astype(int)
data["ACR"] = data["ACR"].fillna(data["ACR"].median())
data["LBXGLU_missing"] = data["LBXGLU"].isna().astype(int)
data["LBXGLU"] = data["LBXGLU"].fillna(data["LBXGLU"].median())
data.isna().sum().sort_values(ascending=False).head()

SEQN        0
RIDAGEYR    0
RIAGENDR    0
RIDRETH3    0
BMXBMI      0
dtype: int64

# Saving Cleaned Data

In [ ]:
data.to_csv("../data/cleaned_data.csv", index = False)